# Topic 7 — Exploratory Data Analysis (EDA)

## Getting to know a dataset before you model it — from first look to decisions

> **Analogy:** EDA is the **doctor's check-up** for a dataset. Before prescribing anything (a model), the doctor measures
> temperature (summary statistics), checks the history (data types, dates), looks for injuries (missing values, impossible
> values, duplicates), and asks "what changed together?" (relationships). Only then is a treatment plan written.

### Roadmap

| Part | Sections |
|---|---|
| 🟢 **Foundations** | 1 What & why · 2 The EDA workflow · 3 The dataset · 4 First look |
| 🟡 **Data quality** | 5 Quality audit · 6 Cleaning *just enough* for EDA |
| 🟡 **One variable** | 7 Numeric columns · 8 Categorical columns · 9 The target |
| 🔴 **Relationships** | 10 Numeric ↔ numeric · 11 Categorical ↔ numeric · 12 Categorical ↔ categorical · 13 Many variables at once |
| 🔴 **Advanced** | 14 Time-based EDA · 15 Is missingness informative? · 16 Leakage red flags · 17 Reusable EDA report function · 18 From findings to decisions · 19 Common mistakes |
| 📝 **Practice** | 18 problems → Solutions |
| 🚀 **Project** | Red-wine quality EDA (real data) |
| 📋 **Reference** | Cheat sheet |

**What you should already know:** Pandas basics (Topic 2), missing values (Topic 3), scaling (Topic 4), outliers (Topic 5)
and encoding (Topic 6). Plots here are kept simple — **Topic 8** teaches visualization in depth.

> 📚 **Course navigation** — you are in **Topic 7: Exploratory Data Analysis (EDA)**
> ⬅️ Previous: Topic 6 — Encoding Techniques  ·  ➡️ Next: Topic 8 — Data Visualization

<details><summary>Full course sequence (click to expand)</summary>

| # | Topic | Notebook |
|---|---|---|
| 01 | Python Basics | `Topic_01_Python_Basics.ipynb` |
| 02 | Pandas DataFrame Basics | `Topic_02_Pandas_DataFrame_Basics.ipynb` |
| 03 | Handling Missing Values | `Topic_03_Handling_Missing_Values.ipynb` |
| 04 | Normalization & Standardization | `Topic_04_Normalization_Standardization.ipynb` |
| 05 | Detecting & Treating Outliers | `Topic_05_Outliers.ipynb` |
| 06 | Encoding Techniques | `Topic_06_Encoding_Techniques.ipynb` |
| **▶ 07 ◀** | **Exploratory Data Analysis (EDA)** | `Topic_07_Exploratory_Data_Analysis.ipynb` |
| 08 | Data Visualization | `Topic_08_Data_Visualization.ipynb` |
| 09 | Pandas Data Manipulation | `Topic_09_Pandas_Data_Manipulation.ipynb` |
| 10 | Combining DataFrames | `Topic_10_Combining_DataFrames.ipynb` |
| 11 | Feature Engineering | `Topic_11_Feature_Engineering.ipynb` |
| 11b | KBinsDiscretizer (Discretization) | `Topic_11b_KBinsDiscretizer.ipynb` |
| 12 | Correlation & Multicollinearity | `Topic_12_Correlation_Multicollinearity.ipynb` |
| 13 | Data Transformation & Skewness | `Topic_13_Data_Transformation_Skewness.ipynb` |
| 14 | Statistics for Data Science | `Topic_14_Statistics_for_Data_Science.ipynb` |
| 15 | Probability for Data Science & ML | `Topic_15_Probability_for_Data_Science_ML.ipynb` |
| 16 | Feature Selection | `Topic_16_Feature_Selection.ipynb` |
| 17 | Machine Learning | `Topic_17_Machine_Learning.ipynb` |

</details>

## 📋 Lesson Index — What This Lesson Covers

This lesson has **21 sections**. Click any one to jump to it.

- [1. What Is EDA and Why Do It?](#sec-1)
- [2. The EDA Workflow (a checklist you can reuse)](#sec-2)
- [3. The Dataset — A Telecom Customer-Churn Table](#sec-3)
- [4. First Look](#sec-4)
- [5. Data-Quality Audit](#sec-5)
- [6. Cleaning *Just Enough* for EDA](#sec-6)
- [7. Univariate Analysis — Numeric Columns](#sec-7)
- [8. Univariate Analysis — Categorical Columns](#sec-8)
- [9. The Target Variable](#sec-9)
- [10. Bivariate — Numeric ↔ Numeric](#sec-10)
- [11. Bivariate — Categorical ↔ Numeric](#sec-11)
- [12. Bivariate — Categorical ↔ Categorical](#sec-12)
- [13. Multivariate Analysis — Many Variables at Once](#sec-13)
- [14. Time-Based EDA](#sec-14)
- [15. Is Missingness Informative?](#sec-15)
- [16. Leakage Red Flags](#sec-16)
- [17. A Reusable EDA Report Function](#sec-17)
- [18. From Findings to Decisions](#sec-18)
- [19. Common EDA Mistakes](#sec-19)
- [🚀 Project — EDA of the Red-Wine Quality Dataset (real data)](#sec-20)
- [📋 EDA Cheat Sheet](#sec-21)

> 🗂️ Full course list: [Topic_00_All_Topics_Index.ipynb](Topic_00_All_Topics_Index.ipynb)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
print('pandas', pd.__version__, '| seaborn', sns.__version__)

pandas 3.0.3 | seaborn 0.13.2


---
<a id="sec-1"></a>
# 1. What Is EDA and Why Do It?

### What
**Exploratory Data Analysis** is the systematic process of summarising, visualising and questioning a dataset to understand
its **structure**, **quality**, **distributions** and **relationships** *before* building models or reports.
The term was popularised by the statistician John Tukey (1977).

### Why
| Without EDA you might… | EDA catches it by… |
|---|---|
| train on a column stored as text (`"1,234.50"`) | checking dtypes |
| let 12 duplicated customers count twice | checking duplicates |
| fill "Age = −1" as a real age | range checks on numeric columns |
| think "lahore" and "Lahore" are different cities | inspecting category values |
| get 99% accuracy from a column that is only filled *after* the outcome | leakage checks |
| use a mean on a very skewed column | looking at the distribution |

### When
- **Always** at the start of a new dataset or a new data delivery.
- Again after major cleaning or merging (did the merge create duplicates?).
- Before choosing scaling (Topic 4), outlier treatment (Topic 5), encoding (Topic 6), transformation (Topic 13) and feature selection (Topic 16).

### When *not* (or when to stop)
EDA has no natural end. Stop when you can answer: *What is each column? How good is the data? What is related to the target?
What do I need to fix?* — and write the answers down. Endless plotting without questions is not EDA.

### EDA vs. confirmatory analysis
| EDA | Confirmatory (Topics 14–15) |
|---|---|
| open questions, find patterns | test a specific hypothesis |
| plots, summaries | p-values, confidence intervals |
| generates hypotheses | checks hypotheses |

---
<a id="sec-2"></a>
# 2. The EDA Workflow (a checklist you can reuse)

```text
 1. Understand the question      → what is the target? what does one row represent?
 2. First look                   → shape, head/sample, dtypes, memory
 3. Data-quality audit           → missing · duplicates · wrong types · impossible values · messy categories · cardinality
 4. Clean just enough            → fix types, standardise labels, remove exact duplicates (keep a raw copy!)
 5. Univariate analysis          → each column alone: centre, spread, shape, outliers, frequencies
 6. Target analysis              → class balance / target distribution
 7. Bivariate analysis           → each feature vs the target, and feature vs feature
 8. Multivariate analysis        → correlation matrix, pivot tables, colour by a third variable
 9. Special checks               → time trends, missingness patterns, leakage
10. Write findings → decisions   → a table of "finding → action" for the cleaning / modelling stage
```

**Golden rules**
1. **Never modify the raw data** — work on a copy (`df = raw.copy()`).
2. **Every plot answers a question.** Write the question first, the answer after.
3. **Numbers + pictures.** A summary table can hide shape; a plot can hide exact values — use both.
4. **In a modelling project, do heavy EDA on the training split** to avoid choosing features by peeking at the test set (see section 16).

---
<a id="sec-3"></a>
# 3. The Dataset — A Telecom Customer-Churn Table

We simulate a realistic **customer churn** dataset (churn = the customer cancels the service). One row = one customer.
It has been made **deliberately messy**, exactly like real exports:

| Column | Meaning |
|---|---|
| `CustomerID` | identifier |
| `Gender`, `Age` | demographics |
| `SignupDate` | first day of service |
| `Tenure` | months as a customer |
| `Contract` | Month-to-month / One year / Two year |
| `InternetService` | DSL / Fiber optic / No |
| `PaymentMethod` | how the customer pays |
| `MonthlyCharges`, `TotalCharges` | billing |
| `SupportCalls` | calls to support in the last 6 months |
| `RefundIssued` | refund paid when the account closed |
| `Churn` | **target**: Yes / No |

Run the cell — you don't need to understand the generator to follow the EDA.

In [2]:
def make_churn_data(n=1000, seed=42):
    rng = np.random.default_rng(seed)
    contract = rng.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.55, 0.25, 0.20])
    tenure = np.where(contract == 'Month-to-month', rng.integers(0, 40, n), rng.integers(0, 72, n))
    internet = rng.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.35, 0.45, 0.20])
    base = pd.Series(internet).map({'DSL': 55, 'Fiber optic': 85, 'No': 25}).to_numpy()
    monthly = (base + rng.normal(0, 10, n)).clip(18).round(2)
    payment = rng.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n, p=[0.35, 0.2, 0.225, 0.225])
    calls = rng.poisson(1.2, n) + (contract == 'Month-to-month') * rng.poisson(0.8, n)
    logit = (-2.2 + 1.4 * (contract == 'Month-to-month') - 0.035 * tenure + 0.025 * (monthly - 60)
             + 0.35 * calls + 0.5 * (payment == 'Electronic check'))
    churn = rng.random(n) < 1 / (1 + np.exp(-logit))
    signup = pd.Timestamp('2025-12-31') - pd.to_timedelta(tenure * 30 + rng.integers(0, 30, n), unit='D')
    total = monthly * tenure * rng.normal(1, 0.03, n)

    df = pd.DataFrame({
        'CustomerID': [f'C{10000 + i}' for i in range(n)],
        'Gender': rng.choice(['Male', 'Female'], n),
        'Age': rng.integers(18, 80, n),
        'SignupDate': signup.strftime('%Y-%m-%d'),
        'Tenure': tenure,
        'Contract': contract,
        'InternetService': internet,
        'PaymentMethod': payment,
        'MonthlyCharges': monthly,
        # stored as TEXT like many CSV exports; brand-new customers (tenure 0) have a blank bill
        'TotalCharges': np.where(tenure == 0, ' ', np.char.mod('%.2f', total)),
        'SupportCalls': calls,
        'RefundIssued': np.where(churn & (rng.random(n) < 0.9), 'Yes', 'No'),   # only known AFTER churn -> leakage!
        'Churn': np.where(churn, 'Yes', 'No'),
    })
    # --- realistic mess ---
    messy = rng.choice(n, 60, replace=False)
    df.loc[messy[:30], 'PaymentMethod'] = df.loc[messy[:30], 'PaymentMethod'].str.lower()
    df.loc[messy[30:], 'PaymentMethod'] = ' ' + df.loc[messy[30:], 'PaymentMethod'] + ' '
    df.loc[rng.choice(n, 5, replace=False), 'Age'] = [-1, -1, 150, 150, 0]
    df.loc[rng.choice(n, 40, replace=False), 'MonthlyCharges'] = np.nan
    df.loc[rng.choice(n, 25, replace=False), 'Gender'] = np.nan
    dups = df.sample(12, random_state=1)
    return pd.concat([df, dups], ignore_index=True)

raw = make_churn_data()
df = raw.copy()          # rule 1: keep the raw data untouched
print('Dataset ready:', df.shape)

Dataset ready: (1012, 13)


---
<a id="sec-4"></a>
# 4. First Look

### What
The first 2 minutes with any dataset: *how big is it, what do rows look like, what types did Pandas infer?*

| Command | Answers |
|---|---|
| `df.shape` | rows × columns |
| `df.head()` / `df.tail()` / `df.sample(5, random_state=0)` | what rows look like (`sample` avoids the "sorted file" trap) |
| `df.info()` | dtypes, non-null counts, memory |
| `df.dtypes` | type of each column |
| `df.nunique()` | how many distinct values per column |
| `df.memory_usage(deep=True)` | real memory per column |

In [ ]:
print('shape:', df.shape)
df.sample(5, random_state=0)

In [ ]:
df.info()

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    'example': df.iloc[0],
})
summary

### What the first look tells us
- `TotalCharges` has dtype **`str`** although it is money → something non-numeric is inside (we'll find blanks `' '`).
- `SignupDate` is **`str`**, not a date → must be converted to use `.dt`.
- `CustomerID` has (almost) one unique value per row → an **identifier**, never a feature.
- `PaymentMethod` has more unique values than the 4 real methods → spelling/whitespace variants.
- `MonthlyCharges` and `Gender` have fewer non-null values than rows → missing values.

> ⚠️ **Common mistake:** jumping to `df.describe()` first. It silently **skips** text columns such as `TotalCharges`,
> so you would never notice that a numeric column was read as text.

---
<a id="sec-5"></a>
# 5. Data-Quality Audit

Check the six classic problems **systematically**, one by one.

| # | Problem | How to detect |
|---|---|---|
| 1 | Missing values | `isna().sum()`, `isna().mean()` |
| 2 | Hidden missing values | blanks `' '`, `'?'`, `'NA'`, `-1`, `0` used as "unknown" |
| 3 | Duplicates | `duplicated().sum()`, duplicated IDs |
| 4 | Wrong data types | numbers/dates stored as text |
| 5 | Impossible / suspicious values | range checks (`Age < 0`, `Age > 110`) |
| 6 | Inconsistent categories | `value_counts()` of every text column, case & whitespace |

In [ ]:
# 1. Missing values
missing = pd.DataFrame({'missing': df.isna().sum(), 'pct': (df.isna().mean() * 100).round(2)})
missing[missing['missing'] > 0]

In [ ]:
# 2. Hidden missing values: text that is blank after stripping spaces
for col in df.select_dtypes(include=['object', 'string']).columns:
    blanks = (df[col].str.strip() == '').sum()
    if blanks:
        print(f'{col}: {blanks} blank strings')

In [ ]:
# 3. Duplicates
print('fully duplicated rows :', df.duplicated().sum())
print('duplicated CustomerIDs:', df['CustomerID'].duplicated().sum())

In [ ]:
# 4. Wrong types: try converting and count what fails
converted = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('TotalCharges values that are not numbers:', converted.isna().sum())
print(df.loc[converted.isna(), 'TotalCharges'].value_counts())

dates = pd.to_datetime(df['SignupDate'], errors='coerce')
print('\nSignupDate values that are not dates:', dates.isna().sum())

In [ ]:
# 5. Impossible values
print(df['Age'].describe().round(1))
print('\nAge outside 18-100:')
print(df.loc[~df['Age'].between(18, 100), ['CustomerID', 'Age']])

In [ ]:
# 6. Inconsistent categories
print(df['PaymentMethod'].value_counts())

### Audit findings
| Problem | Found |
|---|---|
| Missing | `MonthlyCharges` and `Gender` have `NaN`s |
| Hidden missing | `TotalCharges` contains blank strings `' '` (the brand-new customers) |
| Duplicates | 12 exact duplicate rows (same `CustomerID` too) |
| Wrong types | `TotalCharges` is text, `SignupDate` is text |
| Impossible values | a handful of ages: −1, 0 and 150 |
| Inconsistent categories | `PaymentMethod` has lowercase and space-padded variants |

> 💡 Notice we **only recorded** problems so far. Finding and fixing are separate steps — this keeps a clear record of what was wrong.

---
<a id="sec-6"></a>
# 6. Cleaning *Just Enough* for EDA

### What
Fix only what would **distort the exploration**: types, labels, exact duplicates, impossible values.

### Why not clean everything now?
Imputing missing values or removing outliers is a **modelling decision** that should be learned on the training set only
(Topic 3, section 32c). During EDA we keep `NaN` visible and **mark** impossible values as `NaN` instead of guessing them.

In [ ]:
df = raw.copy()

df = df.drop_duplicates()                                                 # exact duplicates
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip().replace('', np.nan))   # blank -> NaN -> float
df['SignupDate'] = pd.to_datetime(df['SignupDate'])
df['PaymentMethod'] = df['PaymentMethod'].str.strip().str.title()         # ' mailed check ' -> 'Mailed Check'
df['Age'] = df['Age'].where(df['Age'].between(18, 100))                   # impossible -> NaN (unknown)
df['Churn_flag'] = (df['Churn'] == 'Yes').astype(int)                     # 0/1 version of the target for averages

print(df.shape)
print(df['PaymentMethod'].value_counts())
print(df.dtypes)

`.str.title()` turned `'Electronic check'` into `'Electronic Check'` — that's fine, as long as **every** row now uses the
same spelling. We now have 4 payment methods, correct types, and no duplicates.

> ⚠️ **Common mistakes**
> - `df['TotalCharges'].astype(float)` → crashes on `' '`. Use `pd.to_numeric(..., errors='coerce')` or replace blanks first.
> - Stripping whitespace but forgetting case (or vice versa).
> - Dropping duplicates by `CustomerID` without first checking the rows are really identical.

---
<a id="sec-7"></a>
# 7. Univariate Analysis — Numeric Columns

### What
Describe **one numeric column at a time**: centre, spread, shape and unusual values.

| Question | Tool |
|---|---|
| Typical value? | mean, **median** |
| How spread out? | std, IQR (Q3 − Q1), min/max |
| Symmetric or skewed? | `skew()` (≈0 symmetric, >0 long right tail, <0 long left tail), histogram |
| Heavy tails / extreme values? | `kurt()`, boxplot, IQR rule (Topic 5) |
| Discrete or continuous? | `nunique()` |

### Why
The shape decides later steps: skewed → consider a log/Box-Cox transform (Topic 13) and median imputation;
many outliers → robust scaler (Topic 4) or capping (Topic 5).

In [ ]:
num_cols = ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportCalls']

desc = df[num_cols].describe().T
desc['median'] = df[num_cols].median()
desc['skew'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurt()
desc['n_unique'] = df[num_cols].nunique()
desc.round(2)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i])
    axes[0, i].set_title(col)
    sns.boxplot(x=df[col], ax=axes[1, i])
plt.tight_layout()
plt.show()

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

pd.Series({c: iqr_outliers(df[c].dropna()) for c in num_cols}, name='IQR outliers')

### Reading the results
- **`TotalCharges`** is strongly **right-skewed** (mean clearly above median): many recent customers with small totals and
  a long tail of loyal customers → a candidate for a log transform (Topic 13).
- **`Tenure`** is spread across 0–71 months, with more short tenures (month-to-month customers).
- **`MonthlyCharges`** is not bell-shaped: the histogram has **several bumps (multimodal)** — hinting that it is really a mix
  of groups (the internet service types). We'll confirm this in section 11.
- **`SupportCalls`** is a small **count** (few unique values) → treat it as discrete; a bar chart suits it better than a histogram.
- `Age` looks roughly uniform after we removed the impossible values.

> ⚠️ **Common mistakes**
> - Reporting only the mean for a skewed column.
> - Calling every IQR "outlier" an error — here the high `TotalCharges` values are simply long-time customers (real, not errors).
> - Using default histogram bins for a discrete count; use `value_counts().sort_index()` instead.

In [ ]:
df['SupportCalls'].value_counts().sort_index().plot(kind='bar', title='SupportCalls (discrete count)', figsize=(7, 3))
plt.show()

---
<a id="sec-8"></a>
# 8. Univariate Analysis — Categorical Columns

| Question | Tool |
|---|---|
| Which levels exist? | `unique()`, `nunique()` (**cardinality**) |
| How frequent is each? | `value_counts()`, `value_counts(normalize=True)` |
| Rare levels? | levels below e.g. 1–5% of rows (Topic 6: group into `Other`) |
| Missing? | `value_counts(dropna=False)` |

In [ ]:
cat_cols = ['Gender', 'Contract', 'InternetService', 'PaymentMethod']

for col in cat_cols:
    freq = pd.DataFrame({'count': df[col].value_counts(dropna=False),
                         'pct': (df[col].value_counts(dropna=False, normalize=True) * 100).round(1)})
    print(f'--- {col} (cardinality = {df[col].nunique()})')
    print(freq, '\n')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, col in zip(axes, cat_cols):
    df[col].value_counts().plot(kind='barh', ax=ax, title=col)
plt.tight_layout()
plt.show()

### Reading the results
- `Contract`: month-to-month is the largest group — important because contract type usually drives churn.
- `PaymentMethod`: after cleaning, exactly 4 levels. Before cleaning there were more (section 5) — **always** re-check cardinality after cleaning.
- `Gender`: about 2.5% missing, shown as `NaN` thanks to `dropna=False`.
- No column has rare levels here, so no grouping into `Other` is needed.

> ⚠️ **Common mistake:** `value_counts()` hides `NaN` by default — pass `dropna=False` during EDA.

---
<a id="sec-9"></a>
# 9. The Target Variable

### Why it deserves its own section
- **Classification:** Is it **balanced**? With 90% "No", a model that always says "No" is 90% accurate — and useless.
  Imbalance affects metric choice (precision/recall/F1 instead of accuracy) and splitting (`stratify=y`) — see Topic 17.
- **Regression:** Is it skewed? A skewed target often benefits from a log transform (Topic 13).

In [ ]:
target = pd.DataFrame({'count': df['Churn'].value_counts(),
                       'pct': (df['Churn'].value_counts(normalize=True) * 100).round(1)})
print(target)
print('\nBaseline accuracy of always predicting the majority class:', target['pct'].max(), '%')

The churners are the **minority class** (about 26% of customers; always predicting "No" is already ~74% accurate). That's moderately imbalanced —
we'll compare **churn rates** (the mean of `Churn_flag`) across groups instead of raw counts from now on.

---
<a id="sec-10"></a>
# 10. Bivariate — Numeric ↔ Numeric

### What
How do two numeric columns move together?

| Tool | Use |
|---|---|
| scatter plot | see the shape (linear? curved? clusters?) |
| `df[a].corr(df[b])` (Pearson) | strength of a **linear** relationship, −1 … +1 |
| `method='spearman'` | strength of a **monotonic** relationship (rank-based, robust to outliers) |
| `df.corr(numeric_only=True)` | all pairs at once |

Correlation is taught in depth in **Topic 12** (including multicollinearity and VIF).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=df, x='Tenure', y='TotalCharges', alpha=0.4, ax=axes[0])
axes[0].set_title('Tenure vs TotalCharges')
sns.scatterplot(data=df, x='Age', y='MonthlyCharges', alpha=0.4, ax=axes[1])
axes[1].set_title('Age vs MonthlyCharges')
plt.tight_layout()
plt.show()

for a, b in [('Tenure', 'TotalCharges'), ('Age', 'MonthlyCharges')]:
    print(f'{a:>6} vs {b:<14} pearson={df[a].corr(df[b]):.3f}  spearman={df[a].corr(df[b], method="spearman"):.3f}')

- `Tenure` and `TotalCharges` are **very strongly related** — by construction, total ≈ monthly × tenure.
  The scatter fans out (a "funnel") because customers pay different monthly amounts.
  Two features carrying almost the same information → **multicollinearity** warning (Topic 12).
- `Age` and `MonthlyCharges`: correlation ≈ 0, a shapeless cloud → no relationship.

> ⚠️ **Correlation ≠ causation**, and correlation ≈ 0 only rules out a *linear* (Pearson) or *monotonic* (Spearman) relationship — always look at the scatter plot.

---
<a id="sec-11"></a>
# 11. Bivariate — Categorical ↔ Numeric

### What
Does a numeric column differ between groups? Compare **group summaries** and **distributions side by side**.

```python
df.groupby('group')['number'].agg(['count', 'mean', 'median', 'std'])
sns.boxplot(data=df, x='group', y='number')
```

In [ ]:
print(df.groupby('InternetService')['MonthlyCharges'].agg(['count', 'mean', 'median', 'std']).round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=df, x='InternetService', y='MonthlyCharges', ax=axes[0])
axes[0].set_title('MonthlyCharges by InternetService')
sns.histplot(data=df, x='MonthlyCharges', hue='InternetService', element='step', ax=axes[1])
axes[1].set_title('The three bumps of MonthlyCharges')
plt.tight_layout()
plt.show()

The multimodal `MonthlyCharges` from section 7 is explained: each internet service has its own price level
(No < DSL < Fiber optic). **A strange univariate shape is often a hidden group** — colour by candidate categories to find it.

### Numeric feature vs a categorical **target**
The most useful question in a churn project: *do churners and non-churners differ?*

In [ ]:
num_vs_target = df.groupby('Churn')[num_cols].median().T
num_vs_target['difference'] = num_vs_target['Yes'] - num_vs_target['No']
print(num_vs_target.round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Tenure', 'MonthlyCharges', 'SupportCalls']):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax, order=['No', 'Yes'])
    ax.set_title(f'{col} by Churn')
plt.tight_layout()
plt.show()

- Churners have **much shorter tenure** — people leave early.
- Churners pay **higher monthly charges** on average.
- Churners make **more support calls** — a classic dissatisfaction signal.
- `Age` shows essentially no difference → probably a weak feature.

---
<a id="sec-12"></a>
# 12. Bivariate — Categorical ↔ Categorical

### What
Count combinations with a **cross-tabulation**, then turn counts into **rates**.

```python
pd.crosstab(df['A'], df['B'])                         # counts
pd.crosstab(df['A'], df['B'], normalize='index')      # each row sums to 1 → rate within each A group
df.groupby('A')['target_flag'].mean()                 # churn rate per group (same idea, one number)
```

> ⚠️ Compare **rates**, not counts. The biggest group almost always has the most churners simply because it is biggest.

In [ ]:
print(pd.crosstab(df['Contract'], df['Churn'], margins=True), '\n')
print((pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100).round(1))

In [ ]:
def churn_rate_table(data, col):
    t = data.groupby(col)['Churn_flag'].agg(customers='count', churn_rate='mean')
    t['churn_rate'] = (t['churn_rate'] * 100).round(1)
    return t.sort_values('churn_rate', ascending=False)

for col in cat_cols:
    print(churn_rate_table(df, col), '\n')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
overall = df['Churn_flag'].mean() * 100
for ax, col in zip(axes, ['Contract', 'PaymentMethod', 'InternetService']):
    churn_rate_table(df, col)['churn_rate'].plot(kind='bar', ax=ax, rot=20, title=f'Churn rate (%) by {col}')
    ax.axhline(overall, color='red', ls='--', label=f'overall {overall:.1f}%')
    ax.legend()
plt.tight_layout()
plt.show()

- **Contract** is the strongest categorical signal: month-to-month customers churn far more than one- and two-year customers.
- **Electronic Check** payers churn more than the other payment methods.
- **Gender** makes almost no difference.
- The dashed line (overall rate) makes it easy to see which groups are above or below average.

A quick statistical check of "is this difference real or noise?" is the **chi-square test** (Topic 14).

---
<a id="sec-13"></a>
# 13. Multivariate Analysis — Many Variables at Once

| Tool | Shows |
|---|---|
| correlation **heatmap** | all numeric pairs at once |
| `pivot_table` with two groupings | target rate for each combination |
| `hue=` in seaborn | a third variable as colour |
| `sns.pairplot` | every numeric pair (use a sample / few columns — it's slow) |

In [ ]:
corr = df[num_cols + ['Churn_flag']].corr(method='spearman')
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Spearman correlation (incl. target)')
plt.show()

In [ ]:
pivot = df.pivot_table(values='Churn_flag', index='Contract', columns='InternetService', aggfunc='mean')
print((pivot * 100).round(1))

plt.figure(figsize=(6.5, 3.5))
sns.heatmap(pivot * 100, annot=True, fmt='.0f', cmap='Reds')
plt.title('Churn rate (%) by Contract × InternetService')
plt.show()

In [ ]:
sns.scatterplot(data=df.sample(500, random_state=0), x='Tenure', y='MonthlyCharges', hue='Churn',
                alpha=0.6, hue_order=['No', 'Yes'])
plt.title('Churners concentrate at low tenure (sample of 500)')
plt.show()

- The heatmap confirms **Tenure ↔ TotalCharges** redundancy and shows `Tenure` (negative) and `SupportCalls` (positive)
  as the numeric columns most correlated with churn.
- The pivot shows how two factors **combine**: month-to-month customers have the highest churn rates within every
  internet type; long contracts are low everywhere.
- The coloured scatter shows churners (orange) concentrated at low tenure.

---
<a id="sec-14"></a>
# 14. Time-Based EDA

### What
When a dataset has dates, ask: *how do things change over time?* and *is the data complete over time?*

```python
df['month'] = df['date'].dt.to_period('M')
df.groupby('month').size()                         # volume over time — gaps? spikes?
df.set_index('date').resample('QE')['x'].mean()    # quarterly averages
```

### Why
- sudden jumps can mean a **data-collection change**, not real behaviour;
- the relationship between features and target may **drift** over time;
- if the data has a time order, a random train/test split can leak the future into the past (Topic 17).

In [ ]:
signups = df.set_index('SignupDate').resample('QE').agg(customers=('CustomerID', 'count'),
                                                        churn_rate=('Churn_flag', 'mean'))
signups['churn_rate'] = (signups['churn_rate'] * 100).round(1)
print(signups.tail(8))

fig, ax1 = plt.subplots(figsize=(12, 4))
signups['customers'].plot(kind='bar', ax=ax1, color='lightsteelblue', label='new customers')
ax1.set_xticklabels([f'{d.year}-Q{d.quarter}' for d in signups.index], rotation=90, fontsize=7)
ax1.set_ylabel('new customers')
ax2 = ax1.twinx()
ax2.plot(range(len(signups)), signups['churn_rate'], color='crimson', marker='o', ms=3, label='churn rate %')
ax2.set_ylabel('churn rate %')
plt.title('Sign-ups per quarter and their churn rate')
plt.tight_layout()
plt.show()

- Recent quarters have **more sign-ups** (month-to-month customers only have tenures up to 39 months, so they are all recent).
- Recent cohorts also show **higher churn rates** — consistent with "short tenure → higher churn".
- Early quarters have few customers, so their churn rate jumps around: **small groups give noisy rates** — always show the count next to a rate.

---
<a id="sec-15"></a>
# 15. Is Missingness Informative?

### What
Compare the target (or other columns) for rows **with** and **without** a missing value.

### Why
If missing rows behave differently, the fact of being missing is itself **information** (MAR/MNAR, Topic 3 section 32b):
keep a `was_missing` flag, and choose the fill value carefully.

In [ ]:
for col in ['MonthlyCharges', 'Gender', 'TotalCharges', 'Age']:
    flag = df[col].isna()
    print(f"{col:<15} missing={flag.sum():>3} | churn rate missing={df.loc[flag, 'Churn_flag'].mean():.2f} "
          f"vs present={df.loc[~flag, 'Churn_flag'].mean():.2f}")

print('\nTenure of rows with missing TotalCharges:', df.loc[df['TotalCharges'].isna(), 'Tenure'].unique())

- `TotalCharges` is missing **only for customers with Tenure = 0** — it isn't random: they simply have no bill yet.
  The right fill is **0**, not the median! EDA found the *reason*, and the reason decides the fix.
  (These 15 brand-new customers also churn more often than average — another sign that "new" matters, captured by `Tenure`.)
- `MonthlyCharges` / `Gender` missing rows have churn rates in the same range as the rest (small groups, so some noise):
  no strong sign that missingness carries information → simple imputation is reasonable.

```python
df['TotalCharges'] = df['TotalCharges'].fillna(0)     # decided by EDA, applies to all data consistently
```

In [ ]:
msno = df[['Age', 'Gender', 'MonthlyCharges', 'TotalCharges']].isna()
plt.figure(figsize=(8, 3))
sns.heatmap(msno.T, cbar=False, cmap='Greys', yticklabels=True, xticklabels=False)
plt.title('Missing-value map (black = missing; each column is a customer)')
plt.show()

---
<a id="sec-16"></a>
# 16. Leakage Red Flags

### What is data leakage?
A feature contains information that **would not be available at prediction time** (usually because it is recorded
*after* the outcome). The model looks brilliant in testing and fails in production.

### EDA red flags
| Red flag | Example |
|---|---|
| a single feature predicts the target almost perfectly | `RefundIssued` = Yes ⇒ churned |
| a column only filled for one class | "cancellation reason" |
| column name suggests the future | `refund`, `closed_date`, `final_*`, `outcome_*` |
| IDs or row numbers correlate with the target | data sorted by outcome before IDs were assigned |

**Question to ask for every column:** *"At the moment I make the prediction, would I already know this value?"*

In [ ]:
print(pd.crosstab(df['RefundIssued'], df['Churn'], normalize='index').round(3))
print('\nChurn rate when RefundIssued = Yes:', round(df.loc[df['RefundIssued'] == 'Yes', 'Churn_flag'].mean(), 3))

**Every** customer with `RefundIssued = Yes` churned — a refund is paid when the account closes, i.e. *after* churn.
It would give a nearly perfect model that is useless in reality → **drop `RefundIssued`** before modelling.

> 💡 A second leakage source is the **analysis process itself**: if you pick features, thresholds or bins by looking at the
> *test* data, your test score is optimistic. In a real project, split first and do target-related EDA on the training split.

---
<a id="sec-17"></a>
# 17. A Reusable EDA Report Function

Doing the same checks on every new dataset? Put them in a function. This one returns one row per column.

In [ ]:
def eda_report(data, target=None):
    rows = []
    for col in data.columns:
        s = data[col]
        row = {'column': col, 'dtype': str(s.dtype), 'missing_pct': round(s.isna().mean() * 100, 2),
               'n_unique': s.nunique(), 'top_value': s.mode().iloc[0] if s.notna().any() else None}
        if pd.api.types.is_numeric_dtype(s):
            row.update(min=s.min(), median=s.median(), max=s.max(), skew=round(s.skew(), 2))
        if target is not None and col != target and pd.api.types.is_numeric_dtype(s) and s.nunique() > 1:
            row['spearman_with_target'] = round(s.corr(data[target], method='spearman'), 3)
        rows.append(row)
    rep = pd.DataFrame(rows).set_index('column')
    rep['flag'] = ''
    rep.loc[rep['missing_pct'] > 20, 'flag'] += 'many-missing '
    rep.loc[rep['n_unique'] == len(data), 'flag'] += 'identifier? '
    rep.loc[rep['n_unique'] <= 1, 'flag'] += 'constant '
    rep.loc[rep.get('skew', pd.Series(dtype=float)).abs() > 1, 'flag'] += 'skewed '
    return rep

eda_report(df, target='Churn_flag')

> 💡 **Automatic profiling tools** such as `ydata-profiling` or `sweetviz` generate a full HTML EDA report in one line.
> They are great for a first overview, but they don't know your business question — you still need sections 9–16.
> (They are not installed in this environment, so they are only mentioned here.)

---
<a id="sec-18"></a>
# 18. From Findings to Decisions

EDA is only finished when the findings become **actions** for the next steps.

| # | Finding | Decision | Where it's taught |
|---|---|---|---|
| 1 | 12 exact duplicate rows | drop them | Topic 2 |
| 2 | `TotalCharges` text with blanks for new customers | convert to number, fill blanks with **0** | Topics 3, 7 |
| 3 | impossible ages (−1, 0, 150) | set to `NaN`, then impute (median, on train) | Topics 3, 5 |
| 4 | `PaymentMethod` spelling variants | strip + standardise case | Topic 9 |
| 5 | `MonthlyCharges` ~4% missing, not informative | median impute (by `InternetService` is even better) | Topic 3 |
| 6 | `TotalCharges` right-skewed | log-transform for linear models | Topic 13 |
| 7 | `Tenure` ↔ `TotalCharges` very highly correlated | keep one, or build `TotalCharges / Tenure` | Topics 11, 12, 16 |
| 8 | `Contract`, `PaymentMethod`, `InternetService` nominal | one-hot encode | Topic 6 |
| 9 | `RefundIssued` leaks the target | **drop** | Topic 7 |
| 10 | `CustomerID` is an identifier | drop from features | Topic 7 |
| 11 | churn is the minority class | stratified split, use recall/F1/ROC-AUC | Topic 17 |
| 12 | strongest signals: Contract, Tenure, SupportCalls, Electronic Check, MonthlyCharges | expect these to matter; `Gender`, `Age` look weak | Topic 16 |

In [ ]:
# Applying the "safe for all data" decisions (no statistics learned from the data here -> no leakage)
model_df = (df.drop(columns=['CustomerID', 'RefundIssued', 'Churn'])
              .assign(TotalCharges=lambda d: d['TotalCharges'].fillna(0)))
print(model_df.shape)
print('remaining missing values (to impute inside the ML pipeline):')
print(model_df.isna().sum()[lambda s: s > 0])

---
<a id="sec-19"></a>
# 19. Common EDA Mistakes

| Mistake | Better |
|---|---|
| Starting with modelling, skipping EDA | do the 10-step workflow first |
| Editing the raw DataFrame | `df = raw.copy()` |
| `describe()` only | it skips text columns — check `dtypes` and `info()` first |
| `value_counts()` without `dropna=False` | missing categories become invisible |
| Comparing counts between groups of different size | compare **rates** (`groupby(...)['flag'].mean()`) and show group size |
| Trusting a rate from 5 rows | show counts; small groups are noisy |
| Only Pearson correlation | also Spearman + scatter plots; relationships can be curved |
| Treating all outliers as errors | ask whether they're real (long-time customers) or impossible (Age 150) |
| Filling missing values without asking *why* they are missing | section 15 — `TotalCharges` needed 0, not the median |
| Keeping post-outcome columns | section 16 — leakage check for every column |
| Target-driven EDA on the full dataset before splitting | split first, explore the training part |
| Lots of plots, no conclusions | write a "finding → decision" table |

---
# Practice Problems (18, increasing difficulty)

You'll run a full EDA on a new, messy **student-performance** dataset. One row = one student.
Target: `FinalScore` (0–100) and its pass/fail version `Passed` (score ≥ 50).

Run the next cell to create `students_raw`. Each problem has an empty code cell; solutions are at the end.

In [ ]:
def make_student_data(n=600, seed=7):
    rng = np.random.default_rng(seed)
    parent = rng.choice(['None', 'High School', 'Bachelor', 'Master'], n, p=[0.15, 0.40, 0.30, 0.15])
    pe_rank = pd.Series(parent).map({'None': 0, 'High School': 1, 'Bachelor': 2, 'Master': 3}).to_numpy()
    internet = rng.choice(['Yes', 'No'], n, p=[0.8, 0.2])
    study = rng.gamma(2, 2.5, n).clip(0, 20).round(1)
    attendance = rng.normal(85, 8, n).clip(40, 100).round(1)
    previous = rng.normal(65, 12, n).clip(0, 100).round(1)
    final = (20 + 0.45 * previous + 1.8 * study + 0.25 * (attendance - 85) + 2 * pe_rank
             + 3 * (internet == 'Yes') + rng.normal(0, 6, n)).clip(0, 100).round(1)
    grade = pd.cut(final, [-1, 49.99, 59.99, 69.99, 79.99, 101], labels=['F', 'D', 'C', 'B', 'A']).astype(str)
    s = pd.DataFrame({
        'StudentID': [f'S{5000 + i}' for i in range(n)],
        'Gender': rng.choice(['Male', 'Female'], n),
        'Age': rng.integers(15, 20, n),
        'ParentEducation': parent,
        'Internet': internet,
        'Club': rng.choice(['Sports', 'Music', 'Debate', 'None'], n),
        'StudyHours': np.char.mod('%.1f', study),        # stored as text
        'Attendance': attendance,
        'PreviousScore': previous,
        'FinalScore': final,
        'GradeLetter': grade,                            # computed FROM FinalScore
    })
    # --- mess ---
    low_att = np.flatnonzero(attendance < 80)
    s.loc[rng.choice(low_att, 30, replace=False), 'StudyHours'] = np.nan   # missing more often for low attendance
    s.loc[rng.choice(n, 12, replace=False), 'StudyHours'] = '?'
    idx = rng.choice(n, 35, replace=False)
    s.loc[idx[:25], 'Internet'] = s.loc[idx[:25], 'Internet'].str.lower()
    s.loc[idx[25:], 'Internet'] = ' ' + s.loc[idx[25:], 'Internet']
    s.loc[rng.choice(n, 3, replace=False), 'Attendance'] = [105.0, 120.0, -5.0]
    s.loc[rng.choice(n, 2, replace=False), 'Age'] = [3, 45]
    s.loc[rng.choice(n, 15, replace=False), 'Gender'] = np.nan
    return pd.concat([s, s.sample(8, random_state=3)], ignore_index=True)

students_raw = make_student_data()
print('students_raw:', students_raw.shape)

### ✍️ Practice Problem 1

**(Easy) First look:** Show the shape, a random sample of 5 rows (`random_state=0`) and `info()`. Which columns have a suspicious dtype?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 1 — write your solution here

### ✍️ Practice Problem 2

**(Easy) Column summary:** Build a table with one row per column: `dtype`, `n_unique`, `missing` count and one example value. Which column is an identifier?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 2 — write your solution here

### ✍️ Practice Problem 3

**(Easy) Missing values:** Show missing count and percentage for every column that has at least one `NaN`.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 3 — write your solution here

### ✍️ Practice Problem 4

**(Easy) Hidden missing & wrong type:** `StudyHours` is text. Find which values are not numbers and how many there are.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 4 — write your solution here

### ✍️ Practice Problem 5

**(Easy) Duplicates:** How many fully duplicated rows are there? How many duplicated `StudentID`s?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 5 — write your solution here

### ✍️ Practice Problem 6

**(Easy) Impossible values:** List rows where `Attendance` is outside 0–100 or `Age` is outside 14–20.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 6 — write your solution here

### ✍️ Practice Problem 7

**(Easy) Messy categories:** Show `value_counts()` of `Internet`. What is wrong with it?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 7 — write your solution here

### ✍️ Practice Problem 8

**(Medium) Clean just enough:** Create `students` from a **copy** of `students_raw`: drop exact duplicates, convert `StudyHours` to numbers (`'?'` → NaN), standardise `Internet`, set impossible `Attendance`/`Age` to NaN, and add `Passed` (1 if `FinalScore` ≥ 50). Print the shape and dtypes. *(Use `students` for all following problems.)*

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 8 — write your solution here

### ✍️ Practice Problem 9

**(Medium) Numeric summary:** For `Age, StudyHours, Attendance, PreviousScore, FinalScore` show `describe()` plus median and skew. Which column is the most skewed, and in which direction?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 9 — write your solution here

### ✍️ Practice Problem 10

**(Medium) Distributions & outliers:** Plot a histogram and a boxplot for `StudyHours` and `Attendance`, and count IQR outliers for every numeric column. Are the `StudyHours` outliers errors?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 10 — write your solution here

### ✍️ Practice Problem 11

**(Medium) Categorical frequencies:** For `Gender, ParentEducation, Internet, Club` print counts and percentages **including missing values**.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 11 — write your solution here

### ✍️ Practice Problem 12

**(Medium) Target analysis:** Describe `FinalScore` (histogram + skew) and the pass rate. What accuracy would a model that always predicts "pass" get?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 12 — write your solution here

### ✍️ Practice Problem 13

**(Medium) Numeric ↔ numeric:** Compute Pearson and Spearman correlation of `StudyHours`, `Attendance` and `PreviousScore` with `FinalScore` and draw a scatter plot of `StudyHours` vs `FinalScore`. Which feature is most related?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 13 — write your solution here

### ✍️ Practice Problem 14

**(Medium) Categorical ↔ numeric:** Compare `FinalScore` across `ParentEducation` levels (count, mean, median) in the **natural order** None → High School → Bachelor → Master, with a boxplot. Is there a trend?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 14 — write your solution here

### ✍️ Practice Problem 15

**(Hard) Categorical ↔ categorical:** Show the pass rate (%), mean `FinalScore` and group size for `Internet` and for `Club`. Do the differences look meaningful, or could they be noise? (Bonus: `scipy.stats.chi2_contingency` on the crosstab — Topic 14.)

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 15 — write your solution here

### ✍️ Practice Problem 16

**(Hard) Multivariate:** Make a pivot table of pass rate by `ParentEducation` × `Internet`, and a Spearman correlation heatmap of the numeric columns plus `FinalScore`.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 16 — write your solution here

### ✍️ Practice Problem 17

**(Hard) Is missingness informative?** Compare `Attendance` and `FinalScore` for students with missing vs present `StudyHours`. Is `StudyHours` missing completely at random?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 17 — write your solution here

### ✍️ Practice Problem 18

**(Hard) Leakage + decisions:** (a) Show why `GradeLetter` must not be used to predict `Passed`. (b) Run `eda_report(students, target='FinalScore')` from section 17. (c) Write a *finding → decision* table (at least 6 rows) for this dataset.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 18 — write your solution here

---
# Practice Solutions

Try each problem first! Solutions run top to bottom (Solution 8 creates `students`, used afterwards).

### Solution 1 — First look

In [ ]:
print(students_raw.shape)
display(students_raw.sample(5, random_state=0))
students_raw.info()
# Suspicious: StudyHours is text (str) although it is a number of hours.

### Solution 2 — Column summary

In [ ]:
pd.DataFrame({'dtype': students_raw.dtypes.astype(str),
              'n_unique': students_raw.nunique(),
              'missing': students_raw.isna().sum(),
              'example': students_raw.iloc[0]})
# StudentID has ~one unique value per row -> identifier, not a feature.

### Solution 3 — Missing values

In [ ]:
m = pd.DataFrame({'missing': students_raw.isna().sum(), 'pct': (students_raw.isna().mean() * 100).round(2)})
m[m['missing'] > 0]

### Solution 4 — Hidden missing & wrong type

In [ ]:
as_num = pd.to_numeric(students_raw['StudyHours'], errors='coerce')
bad = students_raw.loc[as_num.isna(), 'StudyHours']
print(bad.value_counts(dropna=False))
# real NaN values + '?' markers are both "not a number".

### Solution 5 — Duplicates

In [ ]:
print('duplicated rows      :', students_raw.duplicated().sum())
print('duplicated StudentIDs:', students_raw['StudentID'].duplicated().sum())

### Solution 6 — Impossible values

In [ ]:
bad_rows = ~students_raw['Attendance'].between(0, 100) | ~students_raw['Age'].between(14, 20)
students_raw.loc[bad_rows, ['StudentID', 'Age', 'Attendance']]

### Solution 7 — Messy categories

In [ ]:
print(students_raw['Internet'].value_counts())
# The same 2 answers are written 4 ways: 'Yes', 'yes', ' Yes' ... -> case and whitespace must be standardised.

### Solution 8 — Clean just enough

In [ ]:
students = students_raw.copy().drop_duplicates()
students['StudyHours'] = pd.to_numeric(students['StudyHours'], errors='coerce')
students['Internet'] = students['Internet'].str.strip().str.title()
students['Attendance'] = students['Attendance'].where(students['Attendance'].between(0, 100))
students['Age'] = students['Age'].where(students['Age'].between(14, 20))
students['Passed'] = (students['FinalScore'] >= 50).astype(int)
print(students.shape)
print(students['Internet'].value_counts())
students.dtypes

### Solution 9 — Numeric summary

In [ ]:
s_num = ['Age', 'StudyHours', 'Attendance', 'PreviousScore', 'FinalScore']
d = students[s_num].describe().T
d['median'] = students[s_num].median()
d['skew'] = students[s_num].skew()
print(d.round(2))
print('\nMost skewed:', d['skew'].abs().idxmax(), round(d.loc[d['skew'].abs().idxmax(), 'skew'], 2))

`StudyHours` is clearly **right-skewed** (positive skew, mean above median): most students study a few hours, a few study a lot.

### Solution 10 — Distributions & outliers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for j, col in enumerate(['StudyHours', 'Attendance']):
    sns.histplot(students[col], kde=True, ax=axes[0, j]); axes[0, j].set_title(col)
    sns.boxplot(x=students[col], ax=axes[1, j])
plt.tight_layout(); plt.show()

def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
print(pd.Series({c: iqr_outliers(students[c].dropna()) for c in s_num}, name='IQR outliers'))

The high `StudyHours` values (a long right tail) are plausible hard-working students, **not errors** — keep them; a log transform (Topic 13) can reduce the skew if needed.

### Solution 11 — Categorical frequencies

In [ ]:
for col in ['Gender', 'ParentEducation', 'Internet', 'Club']:
    print(pd.DataFrame({'count': students[col].value_counts(dropna=False),
                        'pct': (students[col].value_counts(dropna=False, normalize=True) * 100).round(1)}), '\n')

### Solution 12 — Target analysis

In [ ]:
print(students['FinalScore'].describe().round(2))
print('skew:', round(students['FinalScore'].skew(), 2))
students['FinalScore'].plot(kind='hist', bins=30, title='FinalScore', figsize=(7, 3)); plt.show()
pass_rate = students['Passed'].mean()
print(f'pass rate: {pass_rate:.1%}  ->  always predicting "pass" gives {pass_rate:.1%} accuracy')

### Solution 13 — Numeric ↔ numeric

In [ ]:
rel = pd.DataFrame({m: students[['StudyHours', 'Attendance', 'PreviousScore']].corrwith(students['FinalScore'], method=m)
                    for m in ['pearson', 'spearman']}).round(3)
print(rel.sort_values('spearman', ascending=False))
sns.scatterplot(data=students, x='StudyHours', y='FinalScore', alpha=0.5); plt.show()

### Solution 14 — Categorical ↔ numeric (ordered levels)

In [ ]:
order = ['None', 'High School', 'Bachelor', 'Master']
print(students.groupby('ParentEducation')['FinalScore'].agg(['count', 'mean', 'median']).reindex(order).round(1))
sns.boxplot(data=students, x='ParentEducation', y='FinalScore', order=order); plt.show()

Scores rise clearly from *None* → *High School* → *Bachelor*; *Master* is close to *Bachelor* (slightly lower, and a smaller group). The overall **increasing** pattern supports **ordinal encoding** (Topic 6) — but notice real trends are rarely perfectly monotonic.

### Solution 15 — Categorical ↔ categorical

In [ ]:
from scipy.stats import chi2_contingency

for col in ['Internet', 'Club']:
    t = students.groupby(col).agg(students=('Passed', 'count'), pass_rate=('Passed', 'mean'),
                                  mean_score=('FinalScore', 'mean'))
    t['pass_rate'] = (t['pass_rate'] * 100).round(1)
    p = chi2_contingency(pd.crosstab(students[col], students['Passed']))[1]
    print(t.round(1).sort_values('pass_rate', ascending=False))
    print(f'chi-square p-value = {p:.3f}\n')

Both differences are **small** (about 1 point for Internet, about 5 pass-rate points between clubs), and the chi-square
p-values are far above 0.05: with groups of 120–480 students and a 90% pass rate, differences this size can easily be
**noise**. EDA lesson: don't over-read small gaps between rates — show group sizes and, when in doubt, test (Topic 14).

### Solution 16 — Multivariate

In [ ]:
print((students.pivot_table(values='Passed', index='ParentEducation', columns='Internet', aggfunc='mean')
       .reindex(order) * 100).round(1))
plt.figure(figsize=(6.5, 5))
sns.heatmap(students[s_num].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

### Solution 17 — Is missingness informative?

In [ ]:
miss = students['StudyHours'].isna()
print(students.groupby(miss)[['Attendance', 'FinalScore']].mean().round(1).rename(index={False: 'present', True: 'missing'}))

Students with missing `StudyHours` have clearly **lower attendance** → the missingness is related to another column
(**MAR**), not completely random. Keep a `StudyHours_missing` flag and impute using related columns
(e.g. `KNNImputer`, or median within attendance groups) inside the ML pipeline.

### Solution 18 — Leakage + decisions

In [ ]:
print(pd.crosstab(students['GradeLetter'], students['Passed']))
# GradeLetter is computed FROM FinalScore: F <=> not passed, A-D <=> passed. It is the answer in disguise -> drop it.
eda_report(students, target='FinalScore')

| Finding | Decision |
|---|---|
| 8 exact duplicate rows | drop |
| `StudyHours` is text with `'?'` markers | `to_numeric(errors='coerce')` |
| `StudyHours` missing more for low attendance (MAR) | missing flag + KNN / group-based imputation on the training set |
| impossible Attendance (−5, 105, 120) and Age (3, 45) | set to NaN, impute later |
| `Internet` spelled 4 ways | strip + title case |
| `StudyHours` right-skewed with a long tail | keep values; consider log transform for linear models |
| `ParentEducation` has a natural order and a mostly increasing score trend | ordinal encoding |
| `Club` and `Internet` show no clear effect on passing (differences within noise) | weak candidates — confirm with feature selection (Topic 16) |
| `GradeLetter` is derived from the target | **drop** (leakage) |
| `StudentID` identifier | drop from features |

---
<a id="sec-20"></a>
# 🚀 Project — EDA of the Red-Wine Quality Dataset (real data)

**Dataset:** `data/winequality-red.csv` — 1,599 Portuguese red wines, 11 physico-chemical measurements, and a
`quality` score (0–10) given by wine experts.

**Business question:** *Which measurable properties are associated with higher-quality wine, and what must be fixed before modelling?*

We follow the 10-step workflow from section 2 and finish with a finding → decision table.

In [ ]:
wine_raw = pd.read_csv('data/winequality-red.csv')
wine = wine_raw.copy()
print(wine.shape)
wine.head()

### Step 1–3 · First look and quality audit

In [ ]:
wine.info()
print('\nmissing values :', wine.isna().sum().sum())
print('duplicate rows  :', wine.duplicated().sum())
print('negative values :', (wine.select_dtypes('number') < 0).sum().sum())

- All 12 columns are numeric, no missing values, no negative values.
- There **are duplicate rows**. In this dataset they are most likely different wines with identical lab measurements
  (the data has no ID), so removing them is a judgement call. We drop them for the analysis to avoid counting the same
  measurement pattern twice — and note the decision.

In [ ]:
wine = wine.drop_duplicates().reset_index(drop=True)
print('after dropping duplicates:', wine.shape)

### Step 4 · Target: `quality`

In [ ]:
q = pd.DataFrame({'count': wine['quality'].value_counts().sort_index(),
                  'pct': (wine['quality'].value_counts(normalize=True).sort_index() * 100).round(1)})
print(q)
q['count'].plot(kind='bar', title='Wine quality scores', figsize=(7, 3)); plt.show()

Scores 5 and 6 make up the large majority; very low (3–4) and very high (8) wines are **rare**.
A common modelling choice is a binary target *good = quality ≥ 7*. Let's create it and check the balance.

In [ ]:
wine['good'] = (wine['quality'] >= 7).astype(int)
print(wine['good'].value_counts(normalize=True).round(3))

### Step 5 · Univariate: distributions and skewness

In [ ]:
features = wine.columns.drop(['quality', 'good'])
shape = pd.DataFrame({'mean': wine[features].mean(), 'median': wine[features].median(),
                      'skew': wine[features].skew(),
                      'iqr_outliers': [iqr_outliers(wine[c]) for c in features]}).round(3)
shape.sort_values('skew', ascending=False)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flat, features):
    sns.histplot(wine[col], kde=True, ax=ax); ax.set_title(f'{col} (skew {wine[col].skew():.2f})')
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

### Step 6 · Bivariate: every feature vs quality

In [ ]:
assoc = pd.DataFrame({'spearman_quality': wine[features].corrwith(wine['quality'], method='spearman'),
                      'median_good': wine.loc[wine['good'] == 1, features].median(),
                      'median_not_good': wine.loc[wine['good'] == 0, features].median()}).round(3)
assoc.sort_values('spearman_quality', key=abs, ascending=False)

In [ ]:
top = assoc['spearman_quality'].abs().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, top):
    sns.boxplot(data=wine, x='quality', y=col, ax=ax); ax.set_title(f'{col} by quality')
plt.tight_layout(); plt.show()

### Step 7 · Multivariate: feature ↔ feature correlation

In [ ]:
corr = wine[features].corr(method='spearman')
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, annot_kws={'size': 8})
plt.title('Spearman correlation between wine features'); plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(pairs.abs().sort_values(ascending=False).head(5).round(3))

### Step 8 · Report

In [ ]:
eda_report(wine, target='quality')

### Findings → decisions

| # | Finding | Decision |
|---|---|---|
| 1 | no missing or negative values | no imputation needed |
| 2 | duplicate measurement rows | drop for analysis (documented judgement call) |
| 3 | `quality` concentrated on 5–6, rare extremes | use a binary target (`good` ≥ 7, a minority class) or treat as ordinal; stratified split |
| 4 | `chlorides`, `residual sugar`, `sulphates` and the sulfur-dioxide columns are strongly right-skewed with many IQR outliers | log / Yeo-Johnson transform (Topic 13) or robust scaling (Topic 4); outliers look like real wines, don't delete blindly |
| 5 | `alcohol` has the strongest positive association with quality, `volatile acidity` a clear negative one, `sulphates` positive | expect these as key features (Topic 16) |
| 6 | several acidity/density features are correlated with each other (e.g. `fixed acidity` ↔ `citric acid`/`density`/`pH`, the two sulfur-dioxide columns) | check multicollinearity (Topic 12) before linear models |
| 7 | features on very different scales (density ≈ 1, total sulfur dioxide up to ~290) | scale for distance-based / linear models (Topic 4) |

> Compare these conclusions with the correlation table and boxplots above — every row of the table should be backed by an output.

---
<a id="sec-21"></a>
# 📋 EDA Cheat Sheet

## Workflow
```text
question → first look → quality audit → clean just enough → univariate → target → bivariate → multivariate
        → time / missingness / leakage checks → finding → decision table
```

## First look
```python
df.shape; df.sample(5, random_state=0); df.info(); df.dtypes; df.nunique(); df.memory_usage(deep=True)
```

## Quality audit
```python
df.isna().sum(); df.isna().mean()*100                         # missing
(df[col].str.strip() == '').sum()                             # blank strings
df.duplicated().sum(); df['id'].duplicated().sum()            # duplicates
pd.to_numeric(df[col], errors='coerce').isna().sum()          # text that should be numbers
df.loc[~df['age'].between(0, 110)]                            # impossible values
df[col].value_counts(dropna=False)                            # messy / missing categories
```

## Univariate
```python
df[num].describe().T; df[num].median(); df[num].skew(); df[num].kurt()
sns.histplot(df[c], kde=True); sns.boxplot(x=df[c])
df[cat].value_counts(normalize=True)
```

## Target
```python
df[y].value_counts(normalize=True)     # classification balance -> baseline accuracy
df[y].skew()                           # regression skew
```

## Relationships
```python
df[a].corr(df[b], method='spearman')                          # num ↔ num
df.groupby(cat)[num].agg(['count', 'mean', 'median'])         # cat ↔ num
pd.crosstab(df[c1], df[c2], normalize='index')                # cat ↔ cat (rates)
df.groupby(cat)['target_flag'].mean()                         # target rate per group
df.pivot_table(values='flag', index=c1, columns=c2, aggfunc='mean')
sns.heatmap(df[num].corr(method='spearman'), annot=True)
```

## Special checks
```python
df.set_index('date').resample('QE')['x'].agg(['count', 'mean'])   # time
df.groupby(df[c].isna())['target_flag'].mean()                     # informative missingness
pd.crosstab(df[suspect], df[target], normalize='index')            # leakage: near 0/1 rates
```

## Rules
✔ keep `raw` untouched · ✔ every plot answers a question · ✔ rates not counts, with group sizes ·
✔ Pearson **and** Spearman **and** a scatter · ✔ ask *why* data is missing · ✔ check every column for leakage ·
✔ explore the training split in modelling projects · ✔ end with a finding → decision table

🎉 **You have completed Topic 7 — Exploratory Data Analysis!** Next: **Topic 8 — Data Visualization**.